In [12]:
from pathlib import Path
import json

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import requests

In [13]:
project_root = Path.cwd().parents[0]  

df = pd.read_csv(project_root / "data" / "Seattle_Parks_And_Recreation_Park_Addresses_20260520.csv")

In [3]:
def search_park(api_key,name):
    """get park ratings"""

    #search text query endpoint
    url = "https://places.googleapis.com/v1/places:searchText"

    #define headers to use in API request
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": api_key,
        "X-Goog-FieldMask": "places.displayName,places.rating,places.editorialSummary"
    }
    
    data = {
        "textQuery": f"{name}, Seattle"
    }
    
    #make API request 
    response = requests.post(url, headers=headers, json=data)

    #error handling
    if response.status_code != 200:
        print(f"Error for {name}: {response.text}")
        return None

    #retrieve results
    result = response.json()
    parks = result.get("places", [])

    if not parks:
        print(f"No results for {name}")
        return None
    
    parks = parks[0]

    #append results
    df_data = [{   
       "name": parks.get("displayName", {}).get("text", "N/A"),
       "rating": parks.get("rating", None),
       "summary": parks.get("editorialSummary", {}).get("text", None),
       }]

    df = pd.DataFrame(df_data)
    return df


In [15]:
def search_googlelink(api_key,name):
    """get google map link for each park"""

    #search text query endpoint
    url = "https://places.googleapis.com/v1/places:searchText"

    #define headers to use in API request
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": api_key,
        "X-Goog-FieldMask": "places.displayName,places.googleMapsUri"
    }
    
    data = {
        "textQuery": f"{name}, Seattle"
    }
    
    #make API request 
    response = requests.post(url, headers=headers, json=data)

    #error handling
    if response.status_code != 200:
        print(f"Error for {name}: {response.text}")
        return None

    #retrieve results
    result = response.json()
    parks = result.get("places", [])

    if not parks:
        print(f"No results for {name}")
        return None
    
    parks = parks[0]

    #append results
    df_data = [{   
       "name": parks.get("displayName", {}).get("text", "N/A"),
       "googlelink": parks.get("googleMapsUri", None)
       }]

    df = pd.DataFrame(df_data)
    return df


In [4]:
#commented out to avoid repeated API calls 

'''import os
from google.cloud import secretmanager

def get_secret(project_id: str, secret_id: str) -> str:
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{project_id}/secrets/{secret_id}/versions/latest"
    response = client.access_secret_version(request={"name": name})
    return response.payload.data.decode("utf-8")

PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "grounded-camera-449000-f8")
api_key = get_secret(PROJECT_ID, "GOOGLE_API_KEY")
combined_park_df = pd.DataFrame()
for name in df["Name"]:
    park_info = search_park(api_key, name)
    if park_info is not None:
        
        combined_park_df = pd.concat([combined_park_df, park_info], ignore_index=True)


combined_park_df = combined_park_df.rename(columns={"name":"API_park_name"})
combined_park_df.head(50)

final_df = pd.concat([df, api_df], axis=1)
final_df.head()

final_df.to_csv("parks_with_ratings.csv", index=False)'''

'import os\nfrom google.cloud import secretmanager\n\ndef get_secret(project_id: str, secret_id: str) -> str:\n    client = secretmanager.SecretManagerServiceClient()\n    name = f"projects/{project_id}/secrets/{secret_id}/versions/latest"\n    response = client.access_secret_version(request={"name": name})\n    return response.payload.data.decode("utf-8")\n\nPROJECT_ID = os.environ.get("GCP_PROJECT_ID", "grounded-camera-449000-f8")\napi_key = get_secret(PROJECT_ID, "GOOGLE_API_KEY")\ncombined_park_df = pd.DataFrame()\nfor name in df["Name"]:\n    park_info = search_park(api_key, name)\n    if park_info is not None:\n\n        combined_park_df = pd.concat([combined_park_df, park_info], ignore_index=True)\n\n\ncombined_park_df = combined_park_df.rename(columns={"name":"API_park_name"})\ncombined_park_df.head(50)\n\nfinal_df = pd.concat([df, api_df], axis=1)\nfinal_df.head()\n\nfinal_df.to_csv("parks_with_ratings.csv", index=False)'

In [ ]:
'''import os
from google.cloud import secretmanager

def get_secret(project_id: str, secret_id: str) -> str:
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{project_id}/secrets/{secret_id}/versions/latest"
    response = client.access_secret_version(request={"name": name})
    return response.payload.data.decode("utf-8")

PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "grounded-camera-449000-f8")
api_key = get_secret(PROJECT_ID, "GOOGLE_API_KEY")
park_link_df = pd.DataFrame()
for name in df["Name"]:
    park_info = search_googlelink(api_key, name)
    if park_info is not None:
        
        park_link_df = pd.concat([park_link_df, park_info], ignore_index=True)


park_link_df = park_link_df.rename(columns={"name":"API_park_name"})
park_link_df.head(50)

park_link_df.head()'''

,API_park_name,googlelink
0,Pratt Park & Spraypark,https://maps.google.com/?cid=46340008395803701...
1,12th West & West Howe Park,https://maps.google.com/?cid=86594417990825590...
2,12th Ave. S Viewpoint,https://maps.google.com/?cid=76400209733693546...
3,12th Ave. Square Park,https://maps.google.com/?cid=67674163916561580...
4,14th Avenue NW Boat Ramp,https://maps.google.com/?cid=18284004285215374...


In [ ]:
'''park_link_df.to_csv("parks_with_links.csv", index=False)'''

In [19]:
#load combined dataset with ratings
parks_ratings_df = pd.read_csv(project_root / "data" / "parks_with_ratings.csv")
parks_links_df = pd.read_csv(project_root / "data" / "parks_with_links.csv")
parks_df = parks_ratings_df.merge(parks_links_df, on="API_park_name", how="left")

parks_df.to_csv(project_root / "data" / "parks_with_ratings_links.csv", index=False)

In [23]:
#convert dataframe to geodataframe for merging with other data
parks_gdf = gpd.GeoDataFrame(parks_df, geometry=gpd.points_from_xy(parks_df["X Coord"], parks_df["Y Coord"]), crs="EPSG:4326")

#load park boundaries
park_boundaries = gpd.read_file(project_root / "data" / "Park_Boundary_(outline)_-4632690197521687727.geojson")

#merge park boundaries dataset with parks dataset 
parks_gdf = parks_gdf.merge(park_boundaries[["PMA", "geometry"]]
                            .rename(columns={"geometry": "boundary", "PMA": "PMAID"}), 
                            on="PMAID", how="left")
# Load Seattle neighborhood boundaries 

neighborhoods = gpd.read_file(project_root / "data" / "nma_nhoods_sub.geojson")

#spatial join based on geographic relationship 
parks_with_neighborhood = gpd.sjoin(parks_gdf, neighborhoods, how="left", predicate="intersects")
parks_with_neighborhood.to_csv(project_root / "data" / "parks_with_neighborhoods.csv", index=False)

In [11]:
parks_with_neighborhood.head()

,PMAID,LocID,Name,Address,ZIP Code,X Coord,Y Coord,Location 1,API_park_name,rating,summary,geometry,boundary,index_right,OBJECTID,L_HOOD,S_HOOD,S_HOOD_ALT_NAMES,Shape__Area,Shape__Length
0,390,2404,Pratt Park,1800 S Main St,98144,-122.307257,47.601407,"(47.601407, -122.307257)",Pratt Park & Spraypark,4.3,Small park with partial mountain views offers ...,POINT (-122.30726 47.60141),"POLYGON ((1276998.3 222830.916, 1276997.75 222...",31.0,58.0,Central Area,Atlantic,"Judkins Park, Jackson Place, Colman",2.066499e+07,25426.231163
1,281,2545,12th and Howe Play Park,1200 W Howe St,98119,-122.372985,47.636097,"(47.636097, -122.372985)",12th West & West Howe Park,4.5,NaN,POINT (-122.37298 47.6361),"POLYGON ((1260878 236079.312, 1260877.25 23603...",17.0,44.0,Queen Anne,West Queen Anne,None,1.797499e+07,20631.705729
2,4159,2387,12th Ave S Viewpoint,2821 12TH Ave S,98144,-122.317765,47.577953,"(47.577953, -122.317765)",12th Ave. S Viewpoint,4.3,NaN,POINT (-122.31776 47.57795),"MULTIPOLYGON (((1274111.125 214325.891, 127411...",52.0,79.0,Beacon Hill,North Beacon Hill,Jefferson Park,4.847591e+07,33865.489174
3,4467,2382,12th Ave Square Park,564 12th Ave,98122,-122.316455,47.607427,"(47.607427, -122.316455)",12th Ave. Square Park,3.9,NaN,POINT (-122.31646 47.60743),"POLYGON ((1274631.184 224991.971, 1274599.195 ...",28.0,55.0,Central Area,Minor,"Central District, Squire Park",1.793314e+07,18241.548786
4,4010,2546,14th Ave NW Boat Ramp,4400 14th Ave NW,98107,-122.373536,47.660775,"(47.660775, -122.373536)",14th Avenue NW Boat Ramp,4.3,NaN,POINT (-122.37354 47.66078),"POLYGON ((1260832.762 245052.401, 1260932.754 ...",3.0,30.0,Ballard,West Woodland,None,2.219937e+07,21789.651087
